In [1]:
%load_ext autoreload
%autoreload 2

# NLP Evaluation Metrics Analysis: AI Business Insights

This notebook performs a comprehensive analysis of evaluated business insights generated by various LLM strategies. 

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import statsmodels.api as sm

# Load the dataset
# Adjust path if notebook is run from a different directory
data_path = '../data/bacot_metrics_normalized.pkl'
basic_info_path = '../data/bacot_basic_info.pkl'
try:
    normalized_df = pd.read_pickle(data_path)
    basic_info_df = pd.read_pickle(basic_info_path)
    df = basic_info_df.merge(normalized_df,how='left',on='id')
    print(f"Loaded dataset with {df.shape[0]} rows and {df.shape[1]} columns.")
except FileNotFoundError:
    print(f"Error: Could not find file at {data_path}. Please check the path.")

# Display basic info and first few rows
if 'df' in locals():
    display(df.info())
    display(df.head())


Loaded dataset with 240 rows and 44 columns.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 44 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   id                                         240 non-null    object 
 1   strategy                                   240 non-null    object 
 2   model                                      240 non-null    object 
 3   category                                   240 non-null    object 
 4   department                                 240 non-null    object 
 5   user_question                              240 non-null    object 
 6   selected_kpis                              240 non-null    object 
 7   journal                                    234 non-null    object 
 8   norm_answer_hedge_to_booster_ratio_        240 non-null    float64
 9   norm_recommendations_precision_50_         240 non-nu

None

,id,strategy,model,category,department,user_question,selected_kpis,journal,norm_answer_hedge_to_booster_ratio_,norm_recommendations_precision_50_,...,norm_human_answer_modal_must_freq,norm_answer_modal_could_freq,norm_human_answer_modal_could_freq,norm_answer_modal_will_freq,norm_human_answer_modal_will_freq,norm_answer_modal_may_freq,norm_human_answer_modal_may_freq,norm_answer_modal_might_freq,norm_human_answer_modal_might_freq,overall_quality_index
0,3c360beb-f58a-41a4-88b1-f2f17fb0db34,CoT,kpi_analyser_newcot_pipeline,KPI Trend and Performance Analysis,Customer Care,How did the fluctuating backlog growth rate in...,customer_care_ticket_backlog_growth_rate:\ntim...,Sales - 2024-01-08 : A new process was impleme...,-1.018110,-0.768586,...,-0.064685,-0.552914,-0.552914,-0.851685,-0.851685,-0.564040,-0.564040,-0.434371,-0.434371,-0.562842
1,40d2a628-f6d8-4e09-89c3-b97573578f70,CoT,kpi_analyser_newcot_pipeline,KPI Trend and Performance Analysis,Customer Care,Analyse the quarterly response time and resolu...,customer_care_ticket_agent_response_time_avg:\...,Sales - 2024-01-08 : A new process was impleme...,-1.018110,-0.768586,...,-0.064685,-0.552914,-0.552914,-0.851685,-0.851685,-0.564040,-0.564040,-0.434371,-0.434371,-0.581205
2,4308cd05-2f0a-42b0-b27a-5b7949fc75d8,SymCoT,kpi_analyser_newsymcot_pipeline,KPI Trend and Performance Analysis,Customer Care,What trends in average response and resolution...,customer_care_ticket_agent_response_time_avg:\...,"Sales - 2024-06-03 : Effective immediately, we...",0.327670,0.644100,...,-0.064685,-0.552914,-0.552914,-0.007319,-0.007319,0.717696,0.717696,-0.434371,-0.434371,-0.004803
3,800cb9b2-3f5b-4e88-8193-21045f6dc994,NoCoT,kpi_analyser_nocot_pipeline,KPI Trend and Performance Analysis,Customer Care,What trends in average response and resolution...,customer_care_ticket_agent_response_time_avg:\...,"Sales - 2024-06-03 : Effective immediately, we...",2.025854,-0.768586,...,-0.064685,0.351088,0.351088,-0.353594,-0.353594,3.972528,3.972528,-0.434371,-0.434371,0.896605
4,aaecaff7-05e9-4dd6-b1dd-c902d90866cd,SymCoT,kpi_analyser_newsymcot_pipeline,KPI Trend and Performance Analysis,Customer Care,How did the fluctuating backlog growth rate in...,customer_care_ticket_backlog_growth_rate:\ntim...,Sales - 2024-01-08 : A new process was impleme...,-0.169017,0.076780,...,-0.064685,-0.552914,-0.552914,-0.263248,-0.263248,-0.564040,-0.564040,-0.434371,-0.434371,-0.267190


In [3]:
import re

def to_snake_case(text):
    # Insert underscore between lower-upper transitions and convert to lower
    s1 = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', text)
    return re.sub('([a-z0-9])([A-Z])', r'\1_\2', s1).lower()

## 1. Model Ranking by Overall Quality
Understanding which model strategy (e.g., NoCoT, CoT, SymCoT) performs best according to the composite overall quality index.


In [ ]:
if 'df' in locals():
    # Boxplot for overall quality distribution across models
    title = 'Distribution of Overall Quality Index by Model Strategy'
    fig1 = px.box(
        df, 
        x='strategy', 
        y='overall_quality_index', 
        color='strategy',
        points='all',
        title=title,
        labels={'overall_quality_index': 'Overall Quality Index', 'strategy': 'Strategy'},
        color_discrete_sequence=px.colors.qualitative.Pastel
    )
    fig1.update_layout(showlegend=False)
    fig1.write_html(f"./analysis/01_{to_snake_case(title)}.html")
    fig1.show()

    # Aggregate means for a clear ranked bar chart
    title = 'Average Overall Quality Index by Model Strategy'
    model_quality_agg = df.groupby('strategy')['overall_quality_index'].mean().reset_index().sort_values(by='overall_quality_index', ascending=False)
    fig1_bar = px.bar(
        model_quality_agg,
        x='strategy',
        y='overall_quality_index',
        color='strategy',
        title= title,
        text_auto='.3f'
    )
    fig1_bar.update_layout(showlegend=False)
    fig1_bar.write_html(f"./analysis/02_{to_snake_case(title)}.html")
    fig1_bar.show()


## 2. Department & Category Comparisons
Comparing model performance across various company departments and domains to find areas where models excel or struggle.


In [5]:
if 'df' in locals():
    # Drop NAs in department/category/model if any
    df_clean = df.dropna(subset=['department', 'category', 'strategy', 'overall_quality_index'])
    
    # Sunburst chart to see department vs category breakdown of data
    title = 'Quality Distribution Across Departments, Categories, and Models'
    fig2_sun = px.sunburst(
        df_clean, 
        path=['department', 'category', 'strategy'],
        color='overall_quality_index',
        color_continuous_scale='RdBu',
        title=title
    )
    fig2_sun.write_html(f"./analysis/03_{to_snake_case(title)}.html")
    fig2_sun.show()

    # Department comparison using Box plots
    title = 'Overall Quality Index by Department and Model'
    fig2_box = px.box(
        df_clean,
        x='department',
        y='overall_quality_index',
        color='strategy',
        title=title
    )
    fig2_box.write_html(f"./analysis/04_{to_snake_case(title)}.html")
    fig2_box.show()


## 3. Correlation Heatmap
To understand how various normalized metrics interact with each other.


In [6]:
if 'df' in locals():
    # Select key numerical metrics for the correlation heatmap
    numeric_cols = [
        'overall_quality_index', 'norm_answer_hedge_to_booster_ratio_', 
        'norm_recommendations_precision_50_', 'norm_answer_quantification_density_', 
        'norm_factuality_precision_50_', 'norm_flesch_kincaid_grade',
        'norm_flesch_reading_ease', 'norm_clarity_score_avg', 'norm_f1_bert', 
        'norm_relevance_score_avg', 'norm_reasoning_percentage', 'norm_factuality_score_avg',
        'norm_actionability_score_avg', 'norm_reasoning_score_avg', 'norm_answer_causal_density'
    ]

    # Keep only columns that exist in the dataframe
    cols_to_plot = [col for col in numeric_cols if col in df.columns]

    # Compute correlation matrix
    corr_matrix = df[cols_to_plot].corr()

    # Plot heatmap
    title = 'Correlation Heatmap of Evaluation Metrics'
    fig3 = px.imshow(
        corr_matrix, 
        text_auto='.2f', 
        aspect='auto',
        color_continuous_scale='RdBu_r', 
        title=title,
        width=1200,
        height=1000
    )
    fig3.write_html(f"./analysis/05_{to_snake_case(title)}.html")
    fig3.show()


## 4. Hedge Density vs Factuality Scatter
Investigating whether models that use more hedging (uncertainty markers) are more or less factual.


In [ ]:
if 'df' in locals():
    # Identify the correct factuality metric available in the dataset
    fact_cols = ['norm_factuality_score_avg', 'norm_factuality_precision_50_']
    available_fact_cols = [c for c in fact_cols if c in df.columns]
    
    if 'norm_answer_hedge_density' in df.columns and len(available_fact_cols) > 0:
        target_fact_col = available_fact_cols[0]  # Use the first available
        
        title = f'Hedge Density vs {target_fact_col}'
        fig4 = px.scatter(
            df,
            x='norm_answer_hedge_density',
            y=target_fact_col,
            color='strategy',
            trendline='ols', # requires statsmodels
            hover_data=['department', 'category'] if 'department' in df.columns and 'category' in df.columns else None,
            title=title,
            labels={
                'norm_answer_hedge_density': 'Hedge Density (Normalized)',
                target_fact_col: f'{target_fact_col} (Normalized)'
            },
            opacity=0.7
        )
        fig4.write_html(f"./analysis/06_{to_snake_case(title)}.html")
        fig4.show()
    else:
        print("Required columns for Hedge Density vs Factuality Scatter are missing in the dataset.")


## 5. Human vs Model Gap Analysis
Comparing lexical features and densities between human reference answers and model-generated answers.


In [8]:
if 'df' in locals():
    # Identify matching human vs model columns based on the provided schema
    gap_metrics = [
        ('norm_answer_quantification_density_', 'norm_human_answer_quantification_density_'),
        ('norm_answer_causal_density', 'norm_human_answer_causal_density'),
        ('norm_answer_hedge_density', 'norm_human_answer_hedge_density'),
        ('norm_answer_booster_density', 'norm_human_answer_booster_density'),
        ('norm_answer_modal_should_freq', 'norm_human_answer_modal_should_freq'),
        ('norm_answer_modal_must_freq', 'norm_human_answer_modal_must_freq'),
        ('norm_answer_modal_could_freq', 'norm_human_answer_modal_could_freq'),
        ('norm_answer_modal_will_freq', 'norm_human_answer_modal_will_freq')
    ]

    # Clean label mapping
    def clean_label(text):
        return text.replace('norm_answer_', '').replace('_freq', '').replace('_density', '').replace('_', ' ').title()

    gap_data = []
    gap_radar_labels = []
    
    print("Collecting data for comparative gap analysis...")
    for mod_col, hum_col in gap_metrics:
        # Check if both columns exist
        if mod_col in df.columns and hum_col in df.columns:
            feat_label = clean_label(mod_col)
            gap_radar_labels.append(feat_label)
            
            # Add overall human baseline (average of reference dataset)
            gap_data.append({
                'Feature': feat_label, 
                'Value': df[hum_col].mean(), 
                'Source': 'Human Reference'
            })
            
            # Add each model's mean for the specific lexical feature
            if 'strategy' in df.columns:
                for model_name in df['strategy'].unique():
                    model_mean = df[df['strategy'] == model_name][mod_col].mean()
                    gap_data.append({
                        'Feature': feat_label, 
                        'Value': model_mean, 
                        'Source': f'Model: {model_name}'
                    })
            else:
                gap_data.append({
                    'Feature': feat_label, 
                    'Value': df[mod_col].mean(), 
                    'Source': 'Model Average'
                })

    gap_df = pd.DataFrame(gap_data)
    
    if not gap_df.empty:
        # 1. Grouped Bar chart
        title = 'Human vs Prompt Strategy: Lexical Feature Comparison by Strategy'
        fig6_bar = px.bar(
            gap_df, 
            x='Feature', 
            y='Value', 
            color='Source', 
            barmode='group',
            title=title
        )
        fig6_bar.update_layout(xaxis_tickangle=-45)
        fig6_bar.write_html(f"./analysis/07_{to_snake_case(title)}.html")
        fig6_bar.show()
        
        # 2. Add an optional Radar chart comparing 'Human Reference' vs 'Model Average'
        if 'strategy' in df.columns:
            human_means = [gap_df[(gap_df['Feature'] == f) & (gap_df['Source'] == 'Human Reference')]['Value'].values[0] for f in gap_radar_labels]
            
            fig6_radar = go.Figure()
            # Human Trace
            fig6_radar.add_trace(go.Scatterpolar(
                r=human_means + [human_means[0]],
                theta=gap_radar_labels + [gap_radar_labels[0]],
                fill='toself',
                name='Human Reference',
                line_color='black'
            ))
            
            # Model Traces
            colors = px.colors.qualitative.Plotly
            for i, model_name in enumerate(df['strategy'].unique()):
                model_means = [gap_df[(gap_df['Feature'] == f) & (gap_df['Source'] == f'Model: {model_name}')]['Value'].values[0] for f in gap_radar_labels]
                
                fig6_radar.add_trace(go.Scatterpolar(
                    r=model_means + [model_means[0]],
                    theta=gap_radar_labels + [gap_radar_labels[0]],
                    fill='toself',
                    name=f'Model: {model_name}',
                    line_color=colors[i % len(colors)]
                ))

            title = 'Human vs Prompt Strategy Variants: Average Lexical Feature Gap (Radar Chart)'
            fig6_radar.update_layout(
                polar=dict(radialaxis=dict(visible=True)),
                title=title,
                showlegend=True
            )
            fig6_radar.write_html(f"./analysis/08_{to_snake_case(title)}.html")
            fig6_radar.show()
    else:
        print("No paired human vs model metric columns found for gap analysis.")
